In [1]:
# 
import sys
import subprocess
import json
import textwrap
from pathlib import Path
from datetime import datetime
from xml.sax.saxutils import escape

try:
    import pandas as pd
    from reportlab.lib import colors
    from reportlab.lib.pagesizes import A4
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib.enums import TA_LEFT, TA_CENTER, TA_JUSTIFY
    from reportlab.lib.units import inch
    from reportlab.platypus import (
        SimpleDocTemplate,
        Paragraph,
        Spacer,
        Table,
        TableStyle,
        PageBreak,
        Preformatted,
    )
    from pypdf import PdfReader, PdfWriter
except ImportError:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install",
        "pandas", "reportlab", "pyarrow", "pypdf"
    ])
    import pandas as pd
    from reportlab.lib import colors
    from reportlab.lib.pagesizes import A4
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib.enums import TA_LEFT, TA_CENTER, TA_JUSTIFY
    from reportlab.lib.units import inch
    from reportlab.platypus import (
        SimpleDocTemplate,
        Paragraph,
        Spacer,
        Table,
        TableStyle,
        PageBreak,
        Preformatted,
    )
    from pypdf import PdfReader, PdfWriter

# ============================================================
# 1) PROJECT ROOT DISCOVERY
# ============================================================
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_ROOT = PROJECT_ROOT / "data" / "raw"
BRONZE_ROOT = PROJECT_ROOT / "data" / "bronze"
REPORTS_DIR = PROJECT_ROOT / "reports"
VALIDATION_DIR = REPORTS_DIR / "validation"
SRC_DIR = PROJECT_ROOT / "src"
LOGS_DIR = PROJECT_ROOT / "logs"

OUTPUT_FILE_NAME = "04 Data Profiling and Validation- DM4ML-Group51.pdf"
OUTPUT_PATH = PROJECT_ROOT / OUTPUT_FILE_NAME
TEMP_OUTPUT_PATH = PROJECT_ROOT / "__tmp_04_data_profiling_validation.pdf"

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"RAW_ROOT: {RAW_ROOT}")
print(f"BRONZE_ROOT: {BRONZE_ROOT}")
print(f"VALIDATION_DIR: {VALIDATION_DIR}")
print(f"OUTPUT_PATH: {OUTPUT_PATH}")

# ============================================================
# 2) HELPERS
# ============================================================
def safe_str(x):
    try:
        return str(x)
    except Exception:
        return ""

def rel_path(path):
    try:
        return safe_str(path.relative_to(PROJECT_ROOT))
    except Exception:
        return safe_str(path)

def collect_files(base_dir, patterns):
    results = []
    if not base_dir.exists():
        return results
    for pattern in patterns:
        results.extend(base_dir.rglob(pattern))
    return sorted(set(p for p in results if p.is_file()))

def latest_file(base_dir, patterns):
    files = collect_files(base_dir, patterns)
    if not files:
        return None
    return max(files, key=lambda p: p.stat().st_mtime)

def file_info(path):
    if not path or not path.exists():
        return None
    st = path.stat()
    return {
        "name": path.name,
        "relative_path": rel_path(path),
        "size_kb": round(st.st_size / 1024, 2),
        "modified": datetime.fromtimestamp(st.st_mtime).strftime("%Y-%m-%d %H:%M:%S"),
        "suffix": path.suffix.lower(),
    }

def read_text_preview(path, max_lines=50, max_chars=5000):
    if not path or not path.exists():
        return "File not found."
    try:
        text = path.read_text(encoding="utf-8", errors="ignore")
        lines = text.splitlines()[:max_lines]
        out = "\n".join(lines)
        return out[:max_chars]
    except Exception as e:
        return f"Could not preview file: {e}"

def read_json_preview(path, max_chars=5000):
    if not path or not path.exists():
        return "JSON report not found."
    try:
        with open(path, "r", encoding="utf-8") as f:
            payload = json.load(f)
        pretty = json.dumps(payload, indent=2, ensure_ascii=False)
        return pretty[:max_chars]
    except Exception as e:
        return f"Could not read JSON file: {e}"

def preview_csv(path, n=15):
    if not path or not path.exists():
        return pd.DataFrame({"status": ["File not found"]})
    try:
        return pd.read_csv(path).head(n).fillna("")
    except Exception as e:
        return pd.DataFrame({"status": [f"Could not read CSV: {e}"]})

def build_tree_text(base_path, max_depth=6, max_items=300):
    if not base_path.exists():
        return f"{base_path.name}/ (not found)"
    lines = [f"{base_path.name}/"]
    count = 0

    def walk(path, prefix="", depth=0):
        nonlocal count
        if depth >= max_depth or count >= max_items:
            return
        items = sorted(path.iterdir(), key=lambda p: (p.is_file(), p.name.lower()))
        for idx, item in enumerate(items):
            if count >= max_items:
                break
            connector = "└── " if idx == len(items) - 1 else "├── "
            lines.append(prefix + connector + item.name + ("/" if item.is_dir() else ""))
            count += 1
            if item.is_dir():
                extension = "    " if idx == len(items) - 1 else "│   "
                walk(item, prefix + extension, depth + 1)

    walk(base_path)
    if count >= max_items:
        lines.append("... output truncated ...")
    return "\n".join(lines)

def wrap_block_text(text, width=95):
    wrapped_lines = []
    for line in str(text).splitlines():
        if not line.strip():
            wrapped_lines.append("")
            continue
        wrapped = textwrap.wrap(
            line,
            width=width,
            break_long_words=True,
            break_on_hyphens=True,
            replace_whitespace=False,
            drop_whitespace=False,
        )
        wrapped_lines.extend(wrapped if wrapped else [""])
    return "\n".join(wrapped_lines)

# Real line breaks only at safe separators: \ / _ - =
def wrap_path_for_pdf(value, max_chunk=36):
    if value is None:
        return ""
    text = str(value).strip()
    if not text:
        return ""

    separators = {"\\", "/", "_", "-", "="}
    parts = []
    token = ""

    for ch in text:
        token += ch
        if ch in separators:
            parts.append(token)
            token = ""
    if token:
        parts.append(token)

    lines = []
    current = ""

    for part in parts:
        if len(current) + len(part) <= max_chunk:
            current += part
        else:
            if current:
                lines.append(current)
            if len(part) <= max_chunk:
                current = part
            else:
                subparts = textwrap.wrap(
                    part,
                    width=max_chunk,
                    break_long_words=True,
                    break_on_hyphens=True
                )
                if subparts:
                    lines.extend(subparts[:-1])
                    current = subparts[-1]
                else:
                    current = part

    if current:
        lines.append(current)

    return "<br/>".join(escape(x) for x in lines)

def wrap_general_text_for_pdf(value, max_len=42):
    if value is None:
        return ""
    text = str(value).strip()
    if not text:
        return ""

    words = text.split()
    lines = []
    current = ""

    for word in words:
        candidate = f"{current} {word}".strip()
        if len(candidate) <= max_len:
            current = candidate
        else:
            if current:
                lines.append(current)
            if len(word) > max_len:
                chunks = textwrap.wrap(
                    word,
                    width=max_len,
                    break_long_words=True,
                    break_on_hyphens=True
                )
                if chunks:
                    lines.extend(chunks[:-1])
                    current = chunks[-1]
                else:
                    current = word
            else:
                current = word

    if current:
        lines.append(current)

    return "<br/>".join(escape(x) for x in lines)

def to_para(value, style, kind="general"):
    if kind == "path":
        return Paragraph(wrap_path_for_pdf(value), style)
    return Paragraph(wrap_general_text_for_pdf(value), style)

def make_wrapped_table(data, col_widths=None, header_bg="#D9EAD3", path_cols=None, file_cols=None):
    path_cols = path_cols or []
    file_cols = file_cols or []

    converted = []
    for r, row in enumerate(data):
        row_cells = []
        for c, cell in enumerate(row):
            style = table_header_style if r == 0 else table_cell_style
            if r == 0:
                row_cells.append(Paragraph(escape(str(cell)), style))
            else:
                if c in path_cols or c in file_cols:
                    row_cells.append(to_para(cell, style, kind="path"))
                else:
                    row_cells.append(to_para(cell, style, kind="general"))
        converted.append(row_cells)

    table = Table(converted, colWidths=col_widths, repeatRows=1)
    table.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor(header_bg)),
        ("TEXTCOLOR", (0, 0), (-1, 0), colors.black),
        ("GRID", (0, 0), (-1, -1), 0.5, colors.grey),
        ("VALIGN", (0, 0), (-1, -1), "TOP"),
        ("LEFTPADDING", (0, 0), (-1, -1), 4),
        ("RIGHTPADDING", (0, 0), (-1, -1), 4),
        ("TOPPADDING", (0, 0), (-1, -1), 5),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 5),
    ]))
    return table

def df_to_wrapped_table(df, col_widths=None):
    if df is None or df.empty:
        data = [["No data available"]]
    else:
        tmp = df.copy().fillna("")
        data = [list(tmp.columns)] + tmp.astype(str).values.tolist()
    return make_wrapped_table(data, col_widths=col_widths)

def latest_match_name(base_dir, prefix):
    candidates = collect_files(base_dir, [f"**/{prefix}*.csv"])
    if not candidates:
        return None
    return max(candidates, key=lambda p: p.stat().st_mtime)

def find_validation_code_assets():
    patterns = [
        "**/run_validation.txt",
        "**/*validat*.py",
        "**/*validat*.ipynb",
        "**/*quality*.py",
        "**/*quality*.ipynb",
    ]
    matches = []
    for pattern in patterns:
        matches.extend(PROJECT_ROOT.rglob(pattern))
    cleaned = []
    for p in sorted(set(matches)):
        p_str = safe_str(p).lower()
        if ".ipynb_checkpoints" in p_str:
            continue
        if "/venv/" in p_str or "\\venv\\" in p_str or "/.venv/" in p_str or "\\.venv\\" in p_str:
            continue
        if "/site-packages/" in p_str or "\\site-packages\\" in p_str:
            continue
        cleaned.append(p)
    return cleaned

def read_notebook_code_preview(nb_path, max_code_cells=4, max_chars=4000):
    try:
        with open(nb_path, "r", encoding="utf-8") as f:
            payload = json.load(f)
        parts = []
        cell_count = 0
        for cell in payload.get("cells", []):
            if cell.get("cell_type") != "code":
                continue
            cell_count += 1
            src = cell.get("source", [])
            src = "".join(src) if isinstance(src, list) else str(src)
            src = src.strip()
            if not src:
                continue
            parts.append(f"# Code cell {cell_count}\n{src}")
            if len("\n\n".join(parts)) >= max_chars or cell_count >= max_code_cells:
                break
        out = "\n\n".join(parts).strip()
        return out[:max_chars] if out else "No code cells found."
    except Exception as e:
        return f"Could not parse notebook: {e}"

def read_code_asset_preview(path, max_lines=120, max_chars=6000):
    if not path or not path.exists():
        return "Validation code asset not found."
    if path.suffix.lower() == ".ipynb":
        return read_notebook_code_preview(path, max_code_cells=4, max_chars=max_chars)
    return read_text_preview(path, max_lines=max_lines, max_chars=max_chars)

def append_pdf(base_pdf_path, append_pdf_path, output_pdf_path):
    writer = PdfWriter()

    if base_pdf_path and Path(base_pdf_path).exists():
        base_reader = PdfReader(str(base_pdf_path))
        for page in base_reader.pages:
            writer.add_page(page)

    if append_pdf_path and Path(append_pdf_path).exists():
        append_reader = PdfReader(str(append_pdf_path))
        for page in append_reader.pages:
            writer.add_page(page)

    with open(output_pdf_path, "wb") as f:
        writer.write(f)

# ============================================================
# 3) STYLES
# ============================================================
styles = getSampleStyleSheet()

title_style = ParagraphStyle(
    name="CustomTitle",
    parent=styles["Title"],
    alignment=TA_CENTER,
    fontSize=16,
    leading=20,
    spaceAfter=14,
)

meta_style = ParagraphStyle(
    name="MetaStyle",
    parent=styles["Normal"],
    alignment=TA_LEFT,
    fontSize=10.2,
    leading=13,
    spaceAfter=5,
)

heading_style = ParagraphStyle(
    name="HeadingStyle",
    parent=styles["Heading2"],
    alignment=TA_LEFT,
    fontSize=12,
    leading=15,
    spaceAfter=8,
)

sub_heading_style = ParagraphStyle(
    name="SubHeadingStyle",
    parent=styles["Heading3"],
    alignment=TA_LEFT,
    fontSize=10.4,
    leading=12.5,
    spaceAfter=6,
)

body_style = ParagraphStyle(
    name="BodyStyle",
    parent=styles["BodyText"],
    alignment=TA_JUSTIFY,
    fontSize=10.0,
    leading=14,
    spaceAfter=8,
)

bullet_style = ParagraphStyle(
    name="BulletStyle",
    parent=styles["BodyText"],
    alignment=TA_LEFT,
    fontSize=10.0,
    leading=14,
    leftIndent=14,
    firstLineIndent=-8,
    spaceAfter=4,
)

code_style = ParagraphStyle(
    name="CodeStyle",
    parent=styles["Code"],
    fontName="Courier",
    fontSize=7.0,
    leading=8.4,
)

table_header_style = ParagraphStyle(
    name="TableHeaderStyle",
    parent=styles["BodyText"],
    fontName="Helvetica-Bold",
    fontSize=8.1,
    leading=9.3,
    alignment=TA_LEFT,
)

table_cell_style = ParagraphStyle(
    name="TableCellStyle",
    parent=styles["BodyText"],
    fontName="Helvetica",
    fontSize=7.1,
    leading=8.5,
    alignment=TA_LEFT,
)

# ============================================================
# 4) COLLECT VALIDATION EVIDENCE
# ============================================================
validation_code_assets = find_validation_code_assets()
latest_validation_code = None
for asset in validation_code_assets:
    if asset.name == "run_validation.txt":
        latest_validation_code = asset
        break
if latest_validation_code is None and validation_code_assets:
    latest_validation_code = max(validation_code_assets, key=lambda p: p.stat().st_mtime)

latest_run_validation_txt = latest_file(PROJECT_ROOT, ["**/run_validation.txt"])
latest_validation_json = latest_file(VALIDATION_DIR, ["**/data_quality_report_*.json"])
latest_validation_pdf = latest_file(VALIDATION_DIR, ["**/data_quality_report_*.pdf"])
latest_fix_log = latest_file(VALIDATION_DIR, ["**/fix_log_*.csv"])
latest_summary_initial = latest_file(VALIDATION_DIR, ["**/validation_summary_initial_*.csv"])
latest_summary_revalidated = latest_file(VALIDATION_DIR, ["**/validation_summary_revalidated_*.csv"])
latest_issues_initial = latest_file(VALIDATION_DIR, ["**/validation_issues_initial_*.csv"])
latest_issues_revalidated = latest_file(VALIDATION_DIR, ["**/validation_issues_revalidated_*.csv"])
latest_log_jsonl = latest_file(LOGS_DIR, ["**/validation_log_*.jsonl"])

validation_tree = build_tree_text(VALIDATION_DIR, max_depth=5, max_items=200)

validation_artifact_rows = [["Artifact", "Relative Path", "Last Modified", "Size"]]
for p in [
    latest_run_validation_txt,
    latest_validation_json,
    latest_validation_pdf,
    latest_fix_log,
    latest_summary_initial,
    latest_summary_revalidated,
    latest_issues_initial,
    latest_issues_revalidated,
    latest_log_jsonl,
]:
    if p and p.exists():
        info = file_info(p)
        validation_artifact_rows.append([
            info["name"],
            info["relative_path"],
            info["modified"],
            f"{info['size_kb']} KB",
        ])

if len(validation_artifact_rows) == 1:
    validation_artifact_rows.append(["No validation artifacts found", "-", "-", "-"])

validation_code_rows = [["Validation Asset", "Relative Path", "Last Modified", "Size"]]
for p in validation_code_assets[:15]:
    info = file_info(p)
    validation_code_rows.append([
        info["name"],
        info["relative_path"],
        info["modified"],
        f"{info['size_kb']} KB",
    ])

if len(validation_code_rows) == 1:
    validation_code_rows.append(["No validation code assets found", "-", "-", "-"])

summary_initial_df = preview_csv(latest_summary_initial, n=15)
summary_revalidated_df = preview_csv(latest_summary_revalidated, n=15)
issues_initial_df = preview_csv(latest_issues_initial, n=15)
issues_revalidated_df = preview_csv(latest_issues_revalidated, n=15)
fix_log_df = preview_csv(latest_fix_log, n=15)

validation_code_preview = read_code_asset_preview(latest_validation_code, max_lines=140, max_chars=7000)
run_validation_preview = read_text_preview(latest_run_validation_txt, max_lines=70, max_chars=7000)
validation_json_preview = read_json_preview(latest_validation_json, max_chars=7000)
validation_log_preview = read_text_preview(latest_log_jsonl, max_lines=60, max_chars=7000)

initial_status = "Unknown"
final_status = "Unknown"
for line in run_validation_preview.splitlines():
    if "Initial status:" in line:
        initial_status = line.split("Initial status:")[-1].strip()
    if "Final status:" in line:
        final_status = line.split("Final status:")[-1].strip()

print("\nValidation assets discovered:")
for row in validation_artifact_rows[1:]:
    print("-", row[0], "=>", row[1])

print("\nLatest appended validation PDF:", latest_validation_pdf if latest_validation_pdf else "Not found")
print("Initial status:", initial_status)
print("Final status:", final_status)

# ============================================================
# 5) TABLES
# ============================================================
team_data = [
    ["Team Member Name", "Team Member ID"],
    ["BANSHIDHAR RATH", "2025AE05346"],
    ["JITENDRA KUMAR TIWARI", "2025AE05518"],
    ["KATBA ANKIT CHIMANBHAI", "2025AE05229"],
    ["NAVEEN SURATHU", "2025AE05492"],
]

team_table = make_wrapped_table(
    team_data,
    col_widths=[4.0 * inch, 2.0 * inch]
)

validation_asset_table = make_wrapped_table(
    validation_artifact_rows,
    col_widths=[1.85 * inch, 3.00 * inch, 1.00 * inch, 0.65 * inch],
    path_cols=[1],
    file_cols=[0]
)

validation_code_table = make_wrapped_table(
    validation_code_rows,
    col_widths=[1.70 * inch, 3.15 * inch, 1.00 * inch, 0.65 * inch],
    path_cols=[1],
    file_cols=[0]
)

summary_initial_table = df_to_wrapped_table(
    summary_initial_df,
    col_widths=None
)

summary_revalidated_table = df_to_wrapped_table(
    summary_revalidated_df,
    col_widths=None
)

issues_initial_table = df_to_wrapped_table(
    issues_initial_df,
    col_widths=None
)

issues_revalidated_table = df_to_wrapped_table(
    issues_revalidated_df,
    col_widths=None
)

fix_log_table = df_to_wrapped_table(
    fix_log_df,
    col_widths=None
)

# ============================================================
# 6) BUILD PDF STORY
# ============================================================
story = []

story.append(Paragraph("04 Data Profiling and Validation", title_style))
story.append(Paragraph("<b>Course Name:</b> Data Management for Machine Learning", meta_style))
story.append(Paragraph("<b>Assignment Title:</b> End-to-End Data Management Pipeline for a Recommendation System", meta_style))
story.append(Paragraph("<b>Assignment:</b> Group 51 - Data management for Machine Learning Group 51", meta_style))
story.append(Spacer(1, 10))

story.append(Paragraph("<b>Team Members</b>", heading_style))
story.append(team_table)
story.append(Spacer(1, 14))

story.append(Paragraph("1. Objective", heading_style))
story.append(Paragraph(
    "This report documents the project’s data profiling and validation stage. It captures validation code evidence, generated reports, issue logs, summaries, and the latest available data quality report from the validation folder.",
    body_style
))

story.append(Paragraph("2. Objective Coverage", heading_style))
story.append(Paragraph("• Apply validation checks for missing values, duplicate entries, and schema mismatch.", bullet_style))
story.append(Paragraph("• Apply range and format checks where relevant to dataset rules.", bullet_style))
story.append(Paragraph("• Generate a data quality report summarizing metrics, issues, and remediation evidence.", bullet_style))
story.append(Paragraph("• Append the latest available data quality report PDF from reports/validation.", bullet_style))
story.append(Spacer(1, 10))

story.append(Paragraph("3. Validation Artifact Inventory", heading_style))
story.append(Paragraph(
    "The table below lists the latest validation logs and report artifacts discovered from the project. Long file names and long paths are explicitly wrapped to avoid cut-off text in the PDF.",
    body_style
))
story.append(validation_asset_table)
story.append(Spacer(1, 12))

story.append(Paragraph("4. Validation Code Assets", heading_style))
story.append(Paragraph(
    "These are the validation-related scripts, notebooks, or captured code assets discovered in the project structure.",
    body_style
))
story.append(validation_code_table)
story.append(Spacer(1, 12))

story.append(Paragraph("5. Validation Outcome Summary", heading_style))
story.append(Paragraph(
    f"The latest run preview indicates an initial status of <b>{escape(initial_status)}</b> and a final status of <b>{escape(final_status)}</b> where available.",
    body_style
))
story.append(Spacer(1, 8))

story.append(Paragraph("6. Storage Structure for Validation Reports", heading_style))
story.append(Paragraph(
    "The folder tree below documents the structure of the validation reporting area under reports/validation.",
    body_style
))
story.append(Preformatted(wrap_block_text(validation_tree, width=92), code_style))
story.append(Spacer(1, 12))

story.append(PageBreak())

story.append(Paragraph("7. Automated Validation Code Preview", heading_style))
story.append(Paragraph(
    "The following preview captures the latest validation logic asset discovered in the project.",
    body_style
))
if latest_validation_code:
    story.append(Paragraph(f"<b>Source:</b> {escape(rel_path(latest_validation_code))}", meta_style))
story.append(Preformatted(wrap_block_text(validation_code_preview, width=95), code_style))
story.append(Spacer(1, 12))

story.append(Paragraph("8. Latest run_validation Preview", heading_style))
story.append(Paragraph(
    "Long log and code lines are manually wrapped before rendering to preserve readability.",
    body_style
))
story.append(Preformatted(wrap_block_text(run_validation_preview, width=95), code_style))
story.append(Spacer(1, 12))

story.append(Paragraph("9. Latest Data Quality JSON Preview", heading_style))
story.append(Preformatted(wrap_block_text(validation_json_preview, width=95), code_style))
story.append(Spacer(1, 12))

story.append(PageBreak())

story.append(Paragraph("10. Initial Validation Summary Preview", heading_style))
story.append(summary_initial_table)
story.append(Spacer(1, 12))

story.append(Paragraph("11. Revalidated Summary Preview", heading_style))
story.append(summary_revalidated_table)
story.append(Spacer(1, 12))

story.append(Paragraph("12. Initial Validation Issues Preview", heading_style))
story.append(issues_initial_table)
story.append(Spacer(1, 12))

story.append(PageBreak())

story.append(Paragraph("13. Revalidated Issues Preview", heading_style))
story.append(issues_revalidated_table)
story.append(Spacer(1, 12))

story.append(Paragraph("14. Fix Log Preview", heading_style))
story.append(fix_log_table)
story.append(Spacer(1, 12))

story.append(Paragraph("15. Validation Event Log Preview", heading_style))
story.append(Preformatted(wrap_block_text(validation_log_preview, width=95), code_style))
story.append(Spacer(1, 12))

append_note = (
    f"The latest validation PDF report found at {rel_path(latest_validation_pdf)} will be appended after this section."
    if latest_validation_pdf and latest_validation_pdf.exists()
    else "No existing validation PDF report was found to append."
)
story.append(Paragraph("16. Append Existing Data Quality Report", heading_style))
story.append(Paragraph(escape(append_note), body_style))
story.append(Spacer(1, 12))

story.append(Paragraph("17. Conclusion", heading_style))
story.append(Paragraph(
    "This document consolidates validation code evidence, generated summaries, issue logs, fix logs, and the latest available data quality report into a single submission-ready PDF for the Data Profiling and Validation deliverable.",
    body_style
))

# ============================================================
# 7) BUILD MAIN PDF
# ============================================================
def build_pdf(path):
    doc = SimpleDocTemplate(
        str(path),
        pagesize=A4,
        rightMargin=0.50 * inch,
        leftMargin=0.50 * inch,
        topMargin=0.55 * inch,
        bottomMargin=0.55 * inch,
    )
    doc.build(story)

# ============================================================
# 8) MERGE GENERATED PDF + LATEST VALIDATION PDF
# ============================================================
final_output_used = OUTPUT_PATH

try:
    build_pdf(TEMP_OUTPUT_PATH)

    if latest_validation_pdf and latest_validation_pdf.exists():
        append_pdf(TEMP_OUTPUT_PATH, latest_validation_pdf, OUTPUT_PATH)
    else:
        TEMP_OUTPUT_PATH.replace(OUTPUT_PATH)

    print(f"\nPDF created successfully: {OUTPUT_PATH}")

except PermissionError:
    timestamped_output = PROJECT_ROOT / f"04 Data Profiling and Validation- DM4ML-Group51-{datetime.now().strftime('%Y%m%d_%H%M%S')}.pdf"
    final_output_used = timestamped_output

    if TEMP_OUTPUT_PATH.exists():
        try:
            TEMP_OUTPUT_PATH.unlink()
        except Exception:
            pass

    build_pdf(TEMP_OUTPUT_PATH)

    if latest_validation_pdf and latest_validation_pdf.exists():
        append_pdf(TEMP_OUTPUT_PATH, latest_validation_pdf, timestamped_output)
    else:
        TEMP_OUTPUT_PATH.replace(timestamped_output)

    print("\nOriginal output file is likely open or locked.")
    print(f"Saved alternate file instead: {timestamped_output}")

finally:
    if TEMP_OUTPUT_PATH.exists():
        try:
            TEMP_OUTPUT_PATH.unlink()
        except Exception:
            pass

print(f"\nFinal output path: {final_output_used}")
print(f"Latest appended validation PDF: {latest_validation_pdf if latest_validation_pdf else 'None'}")


PROJECT_ROOT: C:\Users\barath\recomart-pipeline
RAW_ROOT: C:\Users\barath\recomart-pipeline\data\raw
BRONZE_ROOT: C:\Users\barath\recomart-pipeline\data\bronze
VALIDATION_DIR: C:\Users\barath\recomart-pipeline\reports\validation
OUTPUT_PATH: C:\Users\barath\recomart-pipeline\04 Data Profiling and Validation- DM4ML-Group51.pdf

Validation assets discovered:
- run_validation.txt => src\02-validation\run_validation.txt
- data_quality_report_20260429T111342Z.json => reports\validation\data_quality_report_20260429T111342Z.json
- data_quality_report_20260429T111342Z.pdf => reports\validation\data_quality_report_20260429T111342Z.pdf
- fix_log_20260429T111342Z.csv => reports\validation\fix_log_20260429T111342Z.csv
- validation_summary_initial_20260429T111342Z.csv => reports\validation\validation_summary_initial_20260429T111342Z.csv
- validation_summary_revalidated_20260429T111342Z.csv => reports\validation\validation_summary_revalidated_20260429T111342Z.csv
- validation_issues_initial_20260429